In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime, timedelta

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data

from tqdm import tqdm

In [2]:
# remove fragmentation pos if it is less or equal than 4 minutes from previous fragmentation pos (iterative)
def smooth_fragmentation_index(fragmentation_pos, fs=10):

    i = 0
    while len(fragmentation_pos) > i+1:
        diff = fragmentation_pos[i+1] - fragmentation_pos[i]
        if diff<=fs*60*4:
            fragmentation_pos = np.concatenate([fragmentation_pos[:i+1], fragmentation_pos[i+2:]])
        else:
            i+=1
            
    return fragmentation_pos

def sleep_indices(stages, verbose=False):

    stages = stages.dropna()

    hours_data_available = stages.shape[0]/fs/3600
    hours_sleep = sum(stages<5)/fs/3600

    if hours_sleep == 0:
        return [hours_data_available, hours_sleep, 
               np.nan, np.nan, np.nan, np.nan, 
               np.nan, np.nan, np.nan]

    perc_n1 = np.round((sum(stages==3)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_n2 = np.round((sum(stages==2)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_n3 = np.round((sum(stages==1)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_r  = np.round((sum(stages==4)/fs/3600) / hours_sleep * 100, 0).astype('int64')

    # sleep fragmentation index:
    w_or_n1 = np.isin(stages,[5,3])
    deep_sleep = np.isin(stages,[1,2,4])
    fragmentation_shift = np.logical_and(w_or_n1[1:], deep_sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos)
    sfi = np.round(len(fragmentation_pos)/hours_sleep, 1)

    # SFI2, based only on transitions to W from N2, N3 or R
    stages_without_n1 = stages[np.isin(stages,[1,2,4,5])]
    w = np.isin(stages_without_n1,[5])
    deep_sleep = np.isin(stages_without_n1,[1,2,4])
    fragmentation_shift = np.logical_and(w[1:], deep_sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos, fs=10)
    sfi2 = np.round(len(fragmentation_pos)/hours_sleep, 1)
    
    # arousal index, based  on transitions to W from sleep
    w = np.isin(stages,[5])
    sleep = np.isin(stages,[1,2,3,4])
    fragmentation_shift = np.logical_and(w[1:], sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos, fs=10)
    arousal_index = np.round(len(fragmentation_pos)/hours_sleep, 1)

    hours_sleep = np.round(hours_sleep, 1)
    hours_data_available = np.round(hours_data_available, 1)

    if verbose: 
        print(f'Hours of data available:     {hours_data_available:.1f}')
        print(f'Hours of sleep:              {hours_sleep:.1f}')
        print(f'N1, N2, N3, R in %:          {[perc_n1, perc_n2, perc_n3, perc_r]}')
        print(f'Sleep Fragmentation Index:   {sfi}')
        print(f'Sleep Fragmentation Index2:  {sfi2}')
        print(f'Arousal Index:               {arousal_index}')

    return [hours_data_available, hours_sleep, 
           perc_n1, perc_n2, perc_n3, perc_r, 
           sfi, sfi2, arousal_index]
        

In [3]:
files_dir = '/media/ssd2/icu_sleep_sleepstaging/h5'
files = os.listdir(files_dir)
files.sort()


FileNotFoundError: [Errno 2] No such file or directory: '/media/ssd2/icu_sleep_sleepstaging/h5'

In [47]:
sleep_stages_summary = pd.DataFrame()
sleep_stages_summary

""


In [48]:
for file in files:
    
    data, hdr = load_sleep_data(os.path.join(files_dir, file), idx_to_datetime=1)
    fs = hdr['fs']

    dates = pd.unique([x.date() for x in data.index])
    start_dt = min(dates) - timedelta(days=1)
    start_dt = pd.Timestamp(str(start_dt)+' 20:00:00')
    end_dt = max(dates) + timedelta(days=1)
    end_dt = pd.Timestamp(str(end_dt)+' 20:00:00')
    delta = end_dt - start_dt
    all_dates = [start_dt + timedelta(days=i) for i in range(delta.days+1) ]
    
    # loop over all days for this patient:
    for i_day in range(len(all_dates)-1):
        start_dt = all_dates[i_day]
        end_dt = all_dates[i_day+1]

        data_day = data.loc[start_dt : end_dt]
    
        [hours_data_available, hours_sleep, 
               perc_n1, perc_n2, perc_n3, perc_r, 
               sfi, sfi2, arousal_index] = sleep_indices(data_day.stage_pred, verbose=False)
        
        if hours_data_available == 0: continue
            
        sleep_stages_summary_tmp = pd.DataFrame()
        sleep_stages_summary_tmp['Study_ID'] = [hdr['study_id']]
        sleep_stages_summary_tmp['Start'] = [str(start_dt)]
        sleep_stages_summary_tmp['End'] = [str(end_dt)]
        sleep_stages_summary_tmp['Hours_Data'] = [hours_data_available]
        sleep_stages_summary_tmp['Hours_Sleep'] = [hours_sleep]
        sleep_stages_summary_tmp['N1 (%)'] = [perc_n1]
        sleep_stages_summary_tmp['N2 (%)'] = [perc_n2]
        sleep_stages_summary_tmp['N3 (%)'] = [perc_n3]
        sleep_stages_summary_tmp['R (%)'] = [perc_r]
        sleep_stages_summary_tmp['SFI'] = [sfi]
        sleep_stages_summary_tmp['SFI_W'] = [sfi2]
        sleep_stages_summary_tmp['Arousal_Index'] = [arousal_index]

        sleep_stages_summary = pd.concat([sleep_stages_summary, sleep_stages_summary_tmp], axis=0)

In [49]:
# sleep_stages_summary.to_csv('/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/sleep_staging/sleep_stages_indices_raw_SFI.csv', index=False)

In [4]:
sleep_stages_summary = pd.read_csv('/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/sleep_staging/sleep_stages_indices_raw_SFI.csv')

In [5]:
sleep_stages_summary

,Study_ID,Start,End,Hours_Data,Hours_Sleep,N1 (%),N2 (%),N3 (%),R (%),SFI,SFI_W,Arousal_Index
0,1,2018-06-05 20:00:00,2018-06-06 20:00:00,4.4,0.7,100.0,0.0,0.0,0.0,0.0,0.0,5.7
1,1,2018-06-06 20:00:00,2018-06-07 20:00:00,7.9,4.8,90.0,1.0,0.0,10.0,1.3,0.6,2.9
2,2,2018-06-10 20:00:00,2018-06-11 20:00:00,0.2,0.2,0.0,0.0,0.0,100.0,0.0,0.0,0.0
3,2,2018-06-11 20:00:00,2018-06-12 20:00:00,8.6,3.5,2.0,0.0,0.0,97.0,2.5,2.0,2.3
4,3,2018-06-18 20:00:00,2018-06-19 20:00:00,11.4,8.8,48.0,51.0,0.0,1.0,5.2,1.0,2.5
...,...,...,...,...,...,...,...,...,...,...,...,...
388,188,2019-11-14 20:00:00,2019-11-15 20:00:00,24.0,24.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0
389,188,2019-11-15 20:00:00,2019-11-16 20:00:00,7.1,6.1,15.0,9.0,0.0,76.0,2.6,1.6,1.8
390,189,2019-11-12 20:00:00,2019-11-13 20:00:00,1.9,1.9,0.0,0.0,0.0,100.0,0.0,0.0,0.0
391,189,2019-11-13 20:00:00,2019-11-14 20:00:00,24.0,24.0,0.0,25.0,4.0,72.0,0.0,0.0,0.0


In [6]:
sleep_stages_summary.columns

Index(['Study_ID', 'Start', 'End', 'Hours_Data', 'Hours_Sleep', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'SFI_W', 'Arousal_Index'],
      dtype='object')

In [7]:
sel = sleep_stages_summary[sleep_stages_summary['R (%)']<100]
sel = sleep_stages_summary

In [8]:
sel.shape

(393, 12)

In [9]:
sleep_stages_summary[sleep_stages_summary['R (%)']==100].Study_ID.unique().shape

(25,)

In [10]:
sleep_stages_summary[sleep_stages_summary.Study_ID==47]

,Study_ID,Start,End,Hours_Data,Hours_Sleep,N1 (%),N2 (%),N3 (%),R (%),SFI,SFI_W,Arousal_Index
85,47,2019-03-11 20:00:00,2019-03-12 20:00:00,1.7,0.2,100.0,0.0,0.0,0.0,0.0,0.0,5.0
86,47,2019-03-12 20:00:00,2019-03-13 20:00:00,24.0,12.0,13.0,44.0,43.0,0.0,1.8,0.3,0.7
87,47,2019-03-13 20:00:00,2019-03-14 20:00:00,21.9,11.3,6.0,47.0,47.0,0.0,2.0,0.4,0.4


In [11]:
variables = ['Hours_Sleep', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal_Index']

In [12]:
from scipy.stats import sem

In [13]:
df = pd.DataFrame()
df['Mean'] = sel[variables].mean(axis=0)
df['Std'] = sel[variables].std(axis=0)
df['Sem'] = sel[variables].sem(axis=0)

In [14]:
sel[variables].std(axis=0)

Hours_Sleep       7.535294
N1 (%)           30.966794
N2 (%)           25.529974
N3 (%)           19.041984
R (%)            35.681391
SFI               6.736781
Arousal_Index     7.962622
dtype: float64

In [15]:
df.round(2).to_csv('summary_statistics.csv')
df.round(2)

,Mean,Std,Sem
Hours_Sleep,8.18,7.54,0.38
N1 (%),31.04,30.97,1.64
N2 (%),29.60,25.53,1.35
N3 (%),8.55,19.04,1.01
R (%),30.81,35.68,1.89
SFI,2.40,6.74,0.36
Arousal_Index,3.05,7.96,0.42


In [16]:
fig, ax = plt.subplots()
ax.bar(np.arange(7), sel[variables].mean(axis=0), yerr=[np.zeros(7,), sel[variables].std(axis=0)], color='navy')
ax.set_xticklabels(['', 'Sleep \n Duration (h)', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'Sleep \n Fragmentation \n Index', 'Arousal \n Index'], rotation=45)
plt.subplots_adjust(bottom=0.25)
plt.title(f'Sleep Stage Analysis for {len(pd.unique(sel.Study_ID))} ICU patients')
for iV in range(7):
    var_mean = np.round(sel[variables[iV]].mean(),1)
    if var_mean > 5:
        ax.annotate(str(var_mean), xy=(iV, var_mean-3), color='white', ha='center')
    else:
        ax.annotate(str(var_mean), xy=(iV, var_mean+10), color='black', ha='center')


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [82]:
plt.savefig('icu_sleep_stages_summary.pdf')
plt.savefig('icu_sleep_stages_summary.svg')

In [83]:
variables = ['Hours_Sleep', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI']
fig, ax = plt.subplots()
ax.bar(np.arange(6), sel[variables].mean(axis=0), yerr=[np.zeros(6,), sel[variables].std(axis=0)], color='navy')
ax.set_xticklabels(['', 'Sleep \n Duration (h)', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'Sleep \n Fragmentation \n Index'], rotation=45)
plt.subplots_adjust(bottom=0.25)
plt.title(f'Sleep Stage Analysis for {len(pd.unique(sel.Study_ID))} ICU patients')
for iV in range(6):
    var_mean = np.round(sel[variables[iV]].mean(),1)
    if var_mean > 5:
        ax.annotate(str(var_mean), xy=(iV, var_mean-3), color='white', ha='center')
    else:
        ax.annotate(str(var_mean), xy=(iV, var_mean+10), color='black', ha='center')


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:3: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  This is separate from the ipykernel package so we can avoid doing imports until


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [84]:
plt.savefig('icu_sleep_stages_summary2.pdf')
plt.savefig('icu_sleep_stages_summary2.svg')

In [23]:
np.arange(7), sleep_stages_summary[variables].mean(axis=0)

(array([0, 1, 2, 3, 4, 5, 6]),
 Hours_Sleep       8.184478
 N1 (%)           31.036517
 N2 (%)           29.595506
 N3 (%)            8.553371
 R (%)            30.811798
 SFI               2.397753
 Arousal_Index     3.046910
 dtype: float64)

In [24]:
np.arange(7), sleep_stages_summary[variables].std(axis=0)

(array([0, 1, 2, 3, 4, 5, 6]),
 Hours_Sleep       7.535294
 N1 (%)           30.966794
 N2 (%)           25.529974
 N3 (%)           19.041984
 R (%)            35.681391
 SFI               6.736781
 Arousal_Index     7.962622
 dtype: float64)

In [17]:
sleep_stages_summary[['Hours_Sleep', 'N1 (%)',
       'N2 (%)', 'N3 (%)', 'R (%)', 'SFI', 'Arousal_Index']].mean(axis=0).values

array([ 8.18447837, 31.03651685, 29.59550562,  8.55337079, 30.81179775,
        2.39775281,  3.04691011])

In [7]:
plt.figure()
plt.plot(stages.values)
plt.scatter(fragmentation_pos, stages[fragmentation_pos], c='r')
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

NameError: name 'stages' is not defined

In [ ]:
plt.close()